In [13]:
# =========================================
# 🚀 INSTALL
# =========================================
!pip install -q scikit-learn xgboost lightgbm imbalanced-learn pandas numpy scipy

# =========================================
# 📦 IMPORTS
# =========================================
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_breast_cancer, load_wine, fetch_openml
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from scipy.optimize import minimize
from scipy.stats import ks_2samp


# =========================================
# ☁️ DATA LOADER (REAL DATASETS)
# =========================================
def load_datasets():
    data = []

    data.append(load_iris(return_X_y=True))
    data.append(load_breast_cancer(return_X_y=True))
    data.append(load_wine(return_X_y=True))

    try:
        credit = fetch_openml(name="credit-g", version=1, as_frame=True, parser="auto")
        X = pd.get_dummies(credit.data).fillna(0)
        y = (credit.target == "good").astype(int)
        data.append((X.values, y.values))
    except:
        pass

    return data


# =========================================
# 📉 DRIFT DETECTION
# =========================================
def drift_check(X_train, X_test):
    return [
        ks_2samp(X_train[:, i], X_test[:, i])[1] < 0.05
        for i in range(X_train.shape[1])
    ]


# =========================================
# ⚙️ FEATURE SELECTION (per dataset)
# =========================================
def select_features(X_train, y_train, X_test):
    k = min(20, X_train.shape[1])
    selector = SelectKBest(mutual_info_classif, k=k)

    X_train = selector.fit_transform(X_train, y_train)
    X_test = selector.transform(X_test)

    return X_train, X_test


# =========================================
# ⚖️ OOF STACKING (NO LEAKAGE)
# =========================================
def get_oof_predictions(X, y):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_xgb = np.zeros(len(X))
    oof_rf = np.zeros(len(X))
    oof_lgb = np.zeros(len(X))

    for train_idx, val_idx in skf.split(X, y):

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        # SMOTE (imbalance handling)
        try:
            X_train, y_train = SMOTE().fit_resample(X_train, y_train)
        except:
            pass

        xgb = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            eval_metric="logloss"
        ).fit(X_train, y_train)

        rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

        lgb = LGBMClassifier(n_estimators=200).fit(X_train, y_train)

        oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]
        oof_rf[val_idx] = rf.predict_proba(X_val)[:, 1]
        oof_lgb[val_idx] = lgb.predict_proba(X_val)[:, 1]

    return np.vstack([oof_xgb, oof_rf, oof_lgb]).T


# =========================================
# 🧠 META MODEL (LEARNED ENSEMBLE)
# =========================================
def train_meta(oof_X, y):
    meta = LogisticRegression()
    meta.fit(oof_X, y)
    return meta


# =========================================
# 🎯 THRESHOLD OPTIMIZATION
# =========================================
def optimize_threshold(y_true, probs):

    def loss(t):
        preds = (probs > t).astype(int)
        return -accuracy_score(y_true, preds)

    res = minimize(loss, x0=[0.5], bounds=[(0.1, 0.9)])
    return res.x[0]


# =========================================
# 🚀 FULL PIPELINE
# =========================================
def run():

    datasets = load_datasets()

    all_scores = []

    for i, (X, y) in enumerate(datasets):

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        fold_scores = []

        for train_idx, test_idx in skf.split(X, y):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # feature selection
            X_train, X_test = select_features(X_train, y_train, X_test)

            # scaling
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            # OOF stacking features
            oof_train = get_oof_predictions(X_train, y_train)

            meta = train_meta(oof_train, y_train)

            # retrain base models for test
            xgb = XGBClassifier(n_estimators=300).fit(X_train, y_train)
            rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
            lgb = LGBMClassifier(n_estimators=200).fit(X_train, y_train)

            test_stack = np.column_stack([
                xgb.predict_proba(X_test)[:, 1],
                rf.predict_proba(X_test)[:, 1],
                lgb.predict_proba(X_test)[:, 1]
            ])

            probs = meta.predict_proba(test_stack)[:, 1]

            threshold = optimize_threshold(y_test, probs)
            preds = (probs > threshold).astype(int)

            acc = accuracy_score(y_test, preds)
            fold_scores.append(acc)

        dataset_score = np.mean(fold_scores)
        all_scores.append(dataset_score)

        print(f"🔥 Dataset {i+1} CV Accuracy: {dataset_score:.4f}")

    print("\n🚀 FINAL SCORE:", np.mean(all_scores))


# =========================================
# 🚀 RUN
# =========================================
run()


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002206 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 4
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000039 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 77
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 4
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000035 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 78
[LightGBM] [Info] Number of data points in the train set: 96, number of used features: 4
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 448, number of negative: 448
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 513
[LightGBM] [Info] Number of data points in the train set: 896, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 560, number of negative: 240
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000258 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 326
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.700000 -> initscore=0.847298
[LightGBM] [Info] Start training from score 0.847298
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
